In [4]:
import re
import urllib.robotparser
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
URL_INICIAL = urljoin(BASE_URL, "catalogue/page-1.html")
ARQUIVO_SAIDA = "livros_books_toscrape.csv"
SEPARADOR = "|"


def pode_fazer_scraping(url: str) -> bool:
    """Verifica permissao no robots.txt para user-agent generico."""
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(urljoin(BASE_URL, "robots.txt"))
    try:
        rp.read()
        return rp.can_fetch("*", url)
    except Exception:
        # Se nao conseguir ler o robots.txt, segue com cautela.
        return True


def normalizar_preco(preco_texto: str):
    """Converte preco em formato £xx.xx para float."""
    if not preco_texto:
        return None

    texto = preco_texto.replace("Â£", "").replace("£", "").strip()
    try:
        return float(texto)
    except ValueError:
        return None


def extrair_rating(classes):
    """Extrai o rating textual da classe CSS (One, Two, Three...)."""
    opcoes = {"One", "Two", "Three", "Four", "Five"}
    for classe in classes:
        if classe in opcoes:
            return classe
    return None


def preparar_campo(valor):
    if pd.isna(valor):
        return ""

    texto = str(valor)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto.replace(SEPARADOR, " ")


if not pode_fazer_scraping(URL_INICIAL):
    raise SystemExit("Acesso bloqueado pelo robots.txt para esta URL.")

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

url_atual = URL_INICIAL
dados = []

while url_atual:
    try:
        response = requests.get(url_atual, headers=headers, timeout=20)
        response.raise_for_status()
    except requests.RequestException as e:
        raise SystemExit(f"Erro na requisicao HTTP em {url_atual}: {e}")

    soup = BeautifulSoup(response.content, "html.parser")
    livros = soup.select("article.product_pod")

    for livro in livros:
        titulo_tag = livro.select_one("h3 a")
        preco_tag = livro.select_one("p.price_color")
        disponibilidade_tag = livro.select_one("p.instock.availability")
        rating_tag = livro.select_one("p.star-rating")

        titulo = None
        link_livro = None
        if titulo_tag:
            titulo = titulo_tag.get("title") or titulo_tag.get_text(" ", strip=True)
            href = titulo_tag.get("href", "")
            link_livro = urljoin(url_atual, href)

        preco_texto = preco_tag.get_text(strip=True) if preco_tag else None
        preco = normalizar_preco(preco_texto)

        disponibilidade = (
            disponibilidade_tag.get_text(" ", strip=True)
            if disponibilidade_tag
            else None
        )
        rating = extrair_rating(rating_tag.get("class", [])) if rating_tag else None

        dados.append(
            {
                "titulo": titulo,
                "preco_texto": preco_texto,
                "preco": preco,
                "disponibilidade": disponibilidade,
                "rating": rating,
                "url_produto": link_livro,
            }
        )

    proxima = soup.select_one("li.next a")
    url_atual = urljoin(url_atual, proxima["href"]) if proxima else None

df = pd.DataFrame(dados)
if not df.empty:
    df = df.drop_duplicates(subset=["url_produto"]).reset_index(drop=True)

print(f"Total de livros extraidos: {len(df)}")
print(df.head(10))

with open(ARQUIVO_SAIDA, "w", encoding="utf-8-sig", newline="") as arquivo:
    arquivo.write(SEPARADOR.join(df.columns) + "\n")
    for _, linha in df.iterrows():
        valores = [preparar_campo(valor) for valor in linha.tolist()]
        arquivo.write(SEPARADOR.join(valores) + "\n")

print(f"Arquivo salvo em: {ARQUIVO_SAIDA}")

Total de livros extraidos: 1000
                                              titulo preco_texto  preco  \
0                               A Light in the Attic      £51.77  51.77   
1                                 Tipping the Velvet      £53.74  53.74   
2                                         Soumission      £50.10  50.10   
3                                      Sharp Objects      £47.82  47.82   
4              Sapiens: A Brief History of Humankind      £54.23  54.23   
5                                    The Requiem Red      £22.65  22.65   
6  The Dirty Little Secrets of Getting Your Dream...      £33.34  33.34   
7  The Coming Woman: A Novel Based on the Life of...      £17.93  17.93   
8  The Boys in the Boat: Nine Americans and Their...      £22.60  22.60   
9                                    The Black Maria      £52.15  52.15   

  disponibilidade rating                                        url_produto  
0        In stock  Three  https://books.toscrape.com/catalogue/a